In [ ]:
import requests
import time
import random
import hashlib
import smtplib
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- CONFIGURATION ---

# URL de la page à surveiller
URL_TO_CHECK = "https://www.marathondumedoc.com/inscription/"

# Vos coordonnées
EMAIL_DEST = "@gmail.com"
PHONE_NUMBER = "" # Format sans espaces

# Configuration Email (Expéditeur)
# N'oubliez pas d'utiliser un "Mot de passe d'application" Google
EMAIL_SENDER = "pierre.elipse@gmail.com"  # <--- REMPLACEZ PAR VOTRE EMAIL
EMAIL_PASSWORD = ""  # <--- REMPLACEZ PAR VOTRE MDP APP

# Configuration SMS via Passerelle Email
# Choisissez l'opérateur : 'orange', 'sfr', 'bouygues'
OPERATEUR_DEST = "orange"

GATEWAYS = {
    'orange': f'{PHONE_NUMBER}@orange.fr',
    'sfr': f'{PHONE_NUMBER}@sfr.fr',
    'bouygues': f'{PHONE_NUMBER}@bouyguestelecom.fr'
}

# Intervalles de temps (en secondes)
MIN_INTERVAL = 50
MAX_INTERVAL = 70

# --- FONCTIONS ---

def get_content_hash(url):
    """Récupère le HTML et retourne son hash MD5."""
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code == 200:
            content = response.text
            return hashlib.md5(content.encode('utf-8')).hexdigest()
        else:
            print(f"Erreur HTTP : {response.status_code}")
            return None
    except Exception as e:
        print(f"Erreur lors de la récupération de la page : {e}")
        return None

def send_notification(subject, message):
    """Envoie un email et un SMS avec le sujet et le message spécifiés."""
    print(f"Envoi notification : {subject}")

    smtp_server = "smtp.gmail.com"
    smtp_port = 587

    try:
        server = smtplib.SMTP(smtp_server, smtp_port)
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)

        # --- 1. Envoi Email ---
        msg_email = MIMEMultipart()
        msg_email['From'] = EMAIL_SENDER
        msg_email['To'] = EMAIL_DEST
        msg_email['Subject'] = subject
        msg_email.attach(MIMEText(message, 'plain'))

        server.send_message(msg_email)
        print(f"-> Email envoyé à {EMAIL_DEST}")

        # --- 2. Envoi SMS ---
        if OPERATEUR_DEST in GATEWAYS:
            sms_dest = GATEWAYS[OPERATEUR_DEST]
            # Pour le SMS, on concatène sujet et message, car l'objet du mail est souvent ignoré par les passerelles SMS
            sms_body = f"{subject}. {message}"

            msg_sms = MIMEText(sms_body)
            msg_sms['From'] = EMAIL_SENDER
            msg_sms['To'] = sms_dest

            server.send_message(msg_sms)
            print(f"-> SMS envoyé au {PHONE_NUMBER}")
        else:
            print("Opérateur non reconnu pour l'envoi SMS.")

        server.quit()

    except Exception as e:
        print(f"Erreur critique lors de l'envoi : {e}")

# --- PROGRAMME PRINCIPAL ---

def main():
    print(f"--- DÉMARRAGE DU SCRIPT ---")
    print(f"Surveillance de : {URL_TO_CHECK}")

    # 1. Envoi de la notification de démarrage
    send_notification(
        "[MARATHON] Script Démarré",
        "Le script de surveillance est maintenant actif. Vous serez alerté si la page change."
    )

    # 2. Initialisation du hash de référence
    current_hash = get_content_hash(URL_TO_CHECK)

    if current_hash is None:
        print("Erreur critique : Impossible de récupérer la page initiale. Arrêt.")
        return

    print("Référence initiale stockée. Surveillance en cours...")

    # Variable pour mémoriser si le "Heartbeat" de 9h a été envoyé aujourd'hui
    last_heartbeat_date = None

    while True:
        # --- Gestion du temps ---
        now = datetime.now()
        current_date = now.date()

        # --- Vérification Heartbeat 9h00 ---
        # Si on est passé 9h00 et qu'on n'a pas encore envoyé le heartbeat aujourd'hui
        if now.hour >= 9 and last_heartbeat_date != current_date:
            send_notification(
                "[MARATHON] Point de contrôle 9h00",
                "Le script tourne toujours correctement. Pas de changement détecté pour le moment."
            )
            last_heartbeat_date = current_date

        # --- Scrapping de la page ---
        # On attend un temps aléatoire avant de checker pour ne pas se faire bloquer
        sleep_time = random.uniform(MIN_INTERVAL, MAX_INTERVAL)
        # On affiche l'heure actuelle pour le suivi visuel dans la console
        print(f"[{now.strftime('%H:%M:%S')}] Prochaine vérification dans {int(sleep_time)} sec...")
        time.sleep(sleep_time)

        # Récupération du nouveau hash
        new_hash = get_content_hash(URL_TO_CHECK)

        if new_hash and new_hash != current_hash:
            # --- CHANGEMENT DÉTECTÉ ---
            alert_msg = f"La page d'inscription a changé !\nVérifiez immédiatement : {URL_TO_CHECK}"
            send_notification("[MARATHON] ALERTE CHANGEMENT PAGE", alert_msg)

            # Mise à jour du hash pour éviter les doublons
            current_hash = new_hash
        elif new_hash == current_hash:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] Aucun changement.")

if __name__ == "__main__":
    main()

--- DÉMARRAGE DU SCRIPT ---
Surveillance de : https://www.marathondumedoc.com/inscription/
Envoi notification : [MARATHON] Script Démarré
-> Email envoyé à petillion99@gmail.com
-> SMS envoyé au 0695729520
Référence initiale stockée. Surveillance en cours...
Envoi notification : [MARATHON] Point de contrôle 9h00
-> Email envoyé à petillion99@gmail.com
-> SMS envoyé au 0695729520
[17:51:25] Prochaine vérification dans 56 sec...


KeyboardInterrupt: 